## Part 2: Data-engineering questions (P2)

Three questions answered with multi-table joins. Each one ends with the answer, the plots, and short notes on
**grain** (what one row is), **missingness** (what is empty and why) and the **pre-race cutoff** (what is known before the race starts).

In [ ]:
# LOCAL SETUP (removed automatically when the notebooks are merged)
import sys
sys.path.append("../src")
from f1_clean import load_clean, SEED

In [ ]:
import pandas as pd               # tables
import matplotlib.pyplot as plt   # charts
import seaborn as sns             # heatmap (backup section)

clean = load_clean()
base = clean["model_base"]    # pre-race columns + the target `podium`
outcomes = clean["outcomes"]  # post-race columns (status, points...). Never features
tables = clean["tables"]      # all other cleaned tables (races, circuits, drivers...)

### Q1: Which circuits convert a front-row start into a podium most often?

**In plain words:** when a driver starts 1st or 2nd (the front row), how often do they finish in the top 3, at each circuit?
Does that ranking change after 2014? We answer it twice: once with **grid** (where the car really started) and once with
**qualifying position** (where it qualified).

**Why 2014?** In 2014 F1 switched to **hybrid engines** (a small turbo engine plus batteries and electric motors).
One team (Mercedes) became much faster than the rest, and every engine part became limited, which caused many more grid penalties.

**Which years count as "before 2014"?** We make three comparisons, each one for grid and for qualifying (6 in total):

| Before | After | Why |
|---|---|---|
| **2004-2013** | 2014-2024 | **Main, the fairest:** similar length (10 vs 11 seasons), grid penalties exist in both, full qualifying data, similar car reliability |
| 1994-2013 | 2014-2024 | from the first season with qualifying data |
| 1950-2013 | 2014-2024 | all of F1 history |

**Tables and joins (multi-hop):**

```
results ──raceId──► races ──circuitId──► circuits        (which circuit was the race at?)
   │
   └──(raceId, driverId)──► qualifying                     (where did the driver qualify?)
```

- `results ⋈ races ⋈ qualifying` is already done inside `load_clean()` → `model_base`
  (`grid`, `podium`, `year`, `circuitId`, `qual_position`). `load_clean()` also used `status` to remove drivers who did not start,
  removed the Indianapolis 500 and kept one row per driver per race.
- Here we add the last hop: `model_base ⋈ circuits` on `circuitId` (circuit name and country).
- `drivers` and `races` are joined only to show names in an example table; `outcomes` only to explain the result (never as a feature).
- Not used: `sprint_results` (a different, shorter race with its own grid), standings, lap times, pit stops.

In [ ]:
# 1) take only the 3 columns we need from the circuits table
circuits = tables["circuits"][["circuitId", "name", "country"]]
# 2) rename "name" to "circuit" so the column name is clear
circuits = circuits.rename(columns={"name": "circuit"})

# 3) join: every row of base gets the name and country of its circuit
q1 = base.merge(circuits, on="circuitId", how="left", validate="many_to_one")

# 4) safety checks: the join must not add or lose rows, and every race must find its circuit
assert len(q1) == len(base), "the join changed the number of rows"
assert q1["circuit"].notna().all(), "a race has no circuit name"

print("rows (one per driver per race):", len(q1))
print("seasons:", q1["year"].min(), "-", q1["year"].max(), "| races:", q1["raceId"].nunique(), "| circuits:", q1["circuit"].nunique())

#### Grid vs qualifying position

- **Qualifying position** = the result of Saturday's timed session (fastest lap = 1st).
- **Grid** = where the car really started on Sunday.
- **Before 2004 they were the same:** the grid was simply the qualifying order.
- **From 2004** the rules limit engines (and from 2008 gearboxes). A driver who changes one early is moved back on the grid
  (a **penalty**), and the drivers behind move up. A car changed after qualifying starts from the **pit lane** (`grid = 0`).
- **After 2014** every part of the hybrid engine is limited, so penalties became about **twice as common**.

The chart shows this per season (from 1994, the first season with qualifying data, so we can see the "before").

In [ ]:
# keep only rows that have a qualifying position (1994 and later)
with_quali = q1[q1["has_qualifying"]].copy()

# places lost between qualifying and the start (positive = started further back)
with_quali["places_lost"] = with_quali["grid"] - with_quali["qual_position"]

# pushed back = lost 3 or more places, or started from the pit lane (grid 0)
with_quali["pushed_back"] = (with_quali["places_lost"] >= 3) | (with_quali["grid"] == 0)

# per season: number of races, number of pushed-back drivers, and drivers per race
per_year = with_quali.groupby("year").agg(races=("raceId", "nunique"), pushed_back=("pushed_back", "sum"))
per_year["per_race"] = per_year["pushed_back"] / per_year["races"]

plt.figure(figsize=(11, 4))
plt.bar(per_year.index, per_year["per_race"], color="tab:red", label="drivers pushed back per race")
plt.axvline(2003.5, color="grey", linestyle="--", label="2004: first engine rule")
plt.axvline(2013.5, color="black", linestyle="--", label="2014: hybrid engines")
plt.title("Drivers who started 3+ places behind their qualifying position (or from the pit lane)")
plt.xlabel("season")
plt.ylabel("drivers per race")
plt.legend()
plt.show()

**Why 3 places and not 1?** The table counts, per race, the drivers who started 1, 2, 3-4 or 5+ places behind their
qualifying position. Losing **1** place happens about equally often in every span, even before 2004 when there were no
penalties, so it is not a penalty (small recording differences, or someone moved ahead after qualifying). Losing **3 or more**
is zero before 2004 and then grows with the penalty rules. 3 places is also the smallest common real penalty
(for example, blocking another driver in qualifying).

In [ ]:
# three time spans: before penalties, first engine/gearbox rules, hybrid era
with_quali["span"] = "1994-2003"
with_quali.loc[with_quali["year"] >= 2004, "span"] = "2004-13"
with_quali.loc[with_quali["year"] >= 2014, "span"] = "2014-24"

# only drivers who started behind their qualifying position (pit-lane starts have a negative value, so they are left out)
lost = with_quali[with_quali["places_lost"] >= 1].copy()

# put every driver in a group by how many places they lost
lost["places"] = "1"
lost.loc[lost["places_lost"] == 2, "places"] = "2"
lost.loc[lost["places_lost"] >= 3, "places"] = "3-4"
lost.loc[lost["places_lost"] >= 5, "places"] = "5+"

# count the drivers per span and group, then divide each row by the number of races in that span
counts = pd.crosstab(lost["span"], lost["places"])
races_per_span = with_quali.groupby("span")["raceId"].nunique()
places_lost_per_race = counts.div(races_per_span, axis=0)

print("drivers per race who started 1, 2, 3-4 or 5+ places behind their qualifying position:")
places_lost_per_race.round(2)

On the **front row**, every driver who qualified 1st/2nd or started 1st/2nd is one of three cases:
**same** (qualified and started on the front row), **penalty** (qualified on the front row but started further back),
or **moved up** (qualified 3rd or lower but started on the front row because someone ahead was penalised).

In [ ]:
# True / False for every row: qualified 1st or 2nd? started 1st or 2nd?
qualified_front = (with_quali["qual_position"] == 1) | (with_quali["qual_position"] == 2)
started_front = (with_quali["grid"] == 1) | (with_quali["grid"] == 2)

# give every row a readable name for its case
SAME = "qualified 1st/2nd and started 1st/2nd (same)"
PENALTY = "qualified 1st/2nd but started further back (penalty)"
MOVED_UP = "qualified 3rd or lower but started 1st/2nd (moved up)"
with_quali["case"] = "not on the front row"
with_quali.loc[qualified_front & started_front, "case"] = SAME
with_quali.loc[qualified_front & ~started_front, "case"] = PENALTY
with_quali.loc[~qualified_front & started_front, "case"] = MOVED_UP

# only drivers on the front row in at least one of the two, counted per span
front_cases = with_quali[qualified_front | started_front]
case_table = pd.crosstab(front_cases["case"], front_cases["span"])
case_table

In [ ]:
# a few real examples, with driver and race names
drivers = tables["drivers"][["driverId", "forename", "surname"]]
races = tables["races"][["raceId", "name"]].rename(columns={"name": "race"})

examples = front_cases[front_cases["case"] != SAME]
examples = examples.merge(drivers, on="driverId", how="left")
examples = examples.merge(races, on="raceId", how="left")
examples = examples.sort_values("year", ascending=False)
examples[["year", "race", "forename", "surname", "qual_position", "grid", "case"]].head(6)

In [ ]:
# missing data: % of drivers with a known qualifying position, per season
coverage = q1.groupby("year")["has_qualifying"].mean() * 100

plt.figure(figsize=(11, 3.5))
plt.bar(coverage.index, coverage.values, color="tab:blue", label="% of drivers with a known qualifying position")
plt.title("Qualifying data per season: none before 1994, patchy 1996-2002, complete from 2003")
plt.xlabel("season")
plt.ylabel("% of drivers")
plt.legend(loc="upper left")
plt.show()

#### Filling the qualifying gaps with the grid (only before 2004)

Qualifying positions are missing before 1994 and for many races in 1996-2002. Qualifying **did** happen in those years and
**decided the grid**; the dataset just does not record it. Before 2004 there were no grid penalties, so the grid is the
qualifying order. The check below proves it on the years where we have both. So we make a new column `qual_or_grid`:
the qualifying position, and where it is missing **before 2004**, the grid. Missing values from 2004 on are **not** filled,
because from then on the grid can differ from qualifying (penalties).

In [ ]:
# check: in 1994-2003 (before penalties), is the grid the same as the qualifying position?
before_penalties = q1[q1["has_qualifying"] & (q1["year"] <= 2003)]
same_position = before_penalties["grid"] == before_penalties["qual_position"]
print("1994-2003: grid = qualifying position for", round(same_position.mean() * 100, 1), "% of drivers")

qualified_front_old = (before_penalties["qual_position"] == 1) | (before_penalties["qual_position"] == 2)
started_front_old = (before_penalties["grid"] == 1) | (before_penalties["grid"] == 2)
print("front row: qualified 1st/2nd:", qualified_front_old.sum(),
      "| of these, started 1st/2nd:", (qualified_front_old & started_front_old).sum())

# new column: the qualifying position; where it is missing before 2004, use the grid instead
q1["qual_or_grid"] = q1["qual_position"]
fill = q1["qual_or_grid"].isna() & (q1["year"] < 2004)
q1.loc[fill, "qual_or_grid"] = q1.loc[fill, "grid"]

print("rows filled with the grid:", fill.sum())
print("still missing (2004 and later, not filled):", q1["qual_or_grid"].isna().sum())

#### Helper functions (our toolbox)

| Function | What it does |
|---|---|
| `add_era(df, start_year)` | keeps the seasons from `start_year` on and labels them "start_year-2013" or "2014-24" |
| `front_row_rate(df, position_column)` | per circuit and era: **starts** (drivers in position 1 or 2), **podiums**, **rate** = podiums ÷ starts, **enough_starts** (at least `MIN_STARTS`) |
| `overall_rate(df, position_column)` | the same, but all circuits together |
| `plot_top10(ax, table, era, title)` | bar chart of the 10 circuits with the highest rate in one era |
| `compare_eras(table, before)` | circuits with enough starts in **both** eras, and the **Spearman** correlation of their two rankings |
| `slope_chart(ax, rates, spearman, title)` | the rank-change chart: one line per circuit from its rank before to its rank after |

**Spearman** compares two rankings: **1** = exactly the same order, **0** = the old order says nothing about the new order,
**−1** = the order is reversed.

In [ ]:
MIN_STARTS = 10   # a circuit needs at least 10 front-row starts (about 5 races) in an era to be ranked


def add_era(df, start_year):
    """Keep the seasons from start_year on and label each row with its era."""
    part = df[df["year"] >= start_year].copy()
    part["era"] = str(start_year) + "-2013"
    part.loc[part["year"] >= 2014, "era"] = "2014-24"
    return part


def front_row_rate(df, position_column):
    """For every circuit and era: front-row starts, podiums and the rate."""
    # keep only drivers in position 1 or 2 (a missing position is not front row)
    is_front = ((df[position_column] == 1) | (df[position_column] == 2)).fillna(False)
    front = df[is_front]
    # count the starts and add up the podiums (podium is 1 or 0) per circuit and era
    table = front.groupby(["circuit", "era"])["podium"].agg(starts="count", podiums="sum")
    table = table.reset_index()
    # rate = podiums / starts
    table["rate"] = table["podiums"] / table["starts"]
    # enough starts to be ranked?
    table["enough_starts"] = table["starts"] >= MIN_STARTS
    return table


def overall_rate(df, position_column):
    """Front-row starts, podiums and rate per era, all circuits together."""
    is_front = ((df[position_column] == 1) | (df[position_column] == 2)).fillna(False)
    front = df[is_front]
    table = front.groupby("era")["podium"].agg(starts="count", podiums="sum")
    table["rate"] = table["podiums"] / table["starts"]
    return table


def plot_top10(ax, table, era, title):
    """Bar chart of the 10 circuits with the highest rate in one era (only circuits with enough starts)."""
    rows = table[(table["era"] == era) & table["enough_starts"]]
    # sort from low to high and keep the last 10, so the best circuit is drawn at the top
    top10 = rows.sort_values(["rate", "starts"]).tail(10)
    ax.barh(top10["circuit"], top10["rate"] * 100, color="tab:green", label="% of front-row starters on the podium")
    # write "podiums of starts" on each bar
    for i in range(len(top10)):
        ax.text(1, i, str(top10["podiums"].iloc[i]) + " of " + str(top10["starts"].iloc[i]),
                va="center", color="white", fontsize=8)
    ax.set_xlim(0, 100)
    ax.set_title(title)
    ax.set_xlabel("% of front-row starters who finished on the podium")
    ax.legend(loc="lower right", fontsize=8)


def compare_eras(table, before, after="2014-24", min_starts=MIN_STARTS):
    """Rates of the circuits that have enough starts in both eras, and the Spearman correlation of their ranks."""
    rates = table.pivot(index="circuit", columns="era", values="rate")[[before, after]]
    starts = table.pivot(index="circuit", columns="era", values="starts")[[before, after]]
    in_both = (starts[before] >= min_starts) & (starts[after] >= min_starts)
    rates = rates[in_both]
    ranks = rates.rank(ascending=False)   # 1 = highest rate
    spearman = ranks[before].corr(ranks[after])
    return rates, spearman


def slope_chart(ax, rates, spearman, title):
    """Left column = order before, right column = order after, one line per circuit."""
    before = rates.columns[0]
    after = rates.columns[1]
    # position of every circuit in each column (0 = top)
    left = pd.Series(range(len(rates)), index=rates.sort_values(before, ascending=False).index)
    right = pd.Series(range(len(rates)), index=rates.sort_values(after, ascending=False).index)
    for circuit in rates.index:
        if right[circuit] < left[circuit]:
            color = "tab:green"
        elif right[circuit] > left[circuit]:
            color = "tab:red"
        else:
            color = "grey"
        ax.plot([0, 1], [left[circuit], right[circuit]], marker="o", color=color)
        ax.text(-0.03, left[circuit], circuit + " (" + str(round(rates.loc[circuit, before] * 100)) + "%)",
                ha="right", va="center", fontsize=8)
        ax.text(1.03, right[circuit], "(" + str(round(rates.loc[circuit, after] * 100)) + "%) " + circuit,
                ha="left", va="center", fontsize=8)
    ax.plot([], [], color="tab:green", label="better rank after")
    ax.plot([], [], color="tab:red", label="worse rank after")
    ax.plot([], [], color="grey", label="same rank")
    ax.invert_yaxis()                       # best at the top
    ax.set_xlim(-1.2, 2.2)
    ax.set_xticks([0, 1])
    ax.set_xticklabels([before, after])
    ax.set_yticks([])
    ax.set_ylabel("rank (best at the top)")
    ax.set_title(title + "\n" + str(len(rates)) + " circuits, Spearman = " + str(round(spearman, 2)))
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.06), ncol=3, fontsize=8)

#### Minimum number of starts

Some circuits hosted only a few races in an era. A circuit with 2 front-row starts and 2 podiums shows 100%, but that is
2 cars, not a pattern, and it would wrongly top the ranking. So a circuit is ranked only if it has **at least 10 front-row
starts in that era (about 5 races)**: then one car can change the rate by at most 10 points. A later cell checks that
choosing 6, 8 or 15 instead does not change the answer.

In [ ]:
all_seasons = add_era(q1, 1950)
all_grid_table = front_row_rate(all_seasons, "grid")

plt.figure(figsize=(8, 4))
plt.hist(all_grid_table["starts"], bins=30, color="tab:blue", edgecolor="white", label="circuit-eras")
plt.axvline(MIN_STARTS, color="red", linestyle="--", label="minimum = 10 starts")
plt.title("Front-row starts per circuit and era (grid 1st/2nd, 1950-2013 and 2014-24)")
plt.xlabel("front-row starts")
plt.ylabel("number of circuit-eras")
plt.legend()
plt.show()

small = all_grid_table[~all_grid_table["enough_starts"]]
print("circuit-eras with fewer than", MIN_STARTS, "starts:", len(small), "of", len(all_grid_table))
print("examples (these are left out of the ranking):")
small.sort_values("rate", ascending=False).head(5)

#### Main comparison: 2004-2013 vs 2014-2024

The fairest comparison: grid penalties exist in both eras (so grid and qualifying mean the same thing on both sides),
the eras have a similar length, and qualifying data is complete. Top row = grid, bottom row = qualifying.

In [ ]:
fair = add_era(q1, 2004)
grid_fair = front_row_rate(fair, "grid")
qual_fair = front_row_rate(fair, "qual_or_grid")

print("Front row -> podium, all circuits together:")
print("grid:")
display(overall_rate(fair, "grid").loc[["2004-2013", "2014-24"]].round(3))
print("qualifying:")
display(overall_rate(fair, "qual_or_grid").loc[["2004-2013", "2014-24"]].round(3))

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
plot_top10(axes[0, 0], grid_fair, "2004-2013", "Top 10 circuits, started 1st/2nd (grid), 2004-2013")
plot_top10(axes[0, 1], grid_fair, "2014-24", "Top 10 circuits, started 1st/2nd (grid), 2014-2024")
plot_top10(axes[1, 0], qual_fair, "2004-2013", "Top 10 circuits, qualified 1st/2nd, 2004-2013")
plot_top10(axes[1, 1], qual_fair, "2014-24", "Top 10 circuits, qualified 1st/2nd, 2014-2024")
plt.tight_layout()
plt.show()

**Does the ranking change?** We take the circuits with at least 10 front-row starts in **both** eras and rank them in each era.
Each line is one circuit: going up (green) = better rank after 2014, going down (red) = worse.

In [ ]:
grid_rates, grid_spearman = compare_eras(grid_fair, "2004-2013")
qual_rates, qual_spearman = compare_eras(qual_fair, "2004-2013")

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
slope_chart(axes[0], grid_rates, grid_spearman, "Rank change 2004-13 -> 2014-24, started 1st/2nd (grid)")
slope_chart(axes[1], qual_rates, qual_spearman, "Rank change 2004-13 -> 2014-24, qualified 1st/2nd")
plt.tight_layout()
plt.show()

#### All six comparisons

The same calculation for the three "before" windows (1950-2013, 1994-2013, 2004-2013), each with grid and with
qualifying (`qual_or_grid`), and each compared with **2014-24**.

In [ ]:
# compute the 6 comparisons: 3 "before" windows x 2 positions
rows = []
for start_year in [1950, 1994, 2004]:
    data = add_era(q1, start_year)
    before = str(start_year) + "-2013"
    for position_column in ["grid", "qual_or_grid"]:
        overall = overall_rate(data, position_column)
        rates, spearman = compare_eras(front_row_rate(data, position_column), before)
        rows.append([before, position_column, overall.loc[before, "rate"] * 100,
                     overall.loc["2014-24", "rate"] * 100, spearman])

six = pd.DataFrame(rows, columns=["before", "position", "rate before", "rate 2014-24", "spearman"])
six["position"] = six["position"].replace("qual_or_grid", "qualifying")   # nicer name for the charts

**1) The podium rate: each "before" window next to 2014-24.** One chart per window. In each chart the grey bar is the
window before 2014 and the green bar is 2014-24, for grid and for qualifying.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, before in zip(axes, ["1950-2013", "1994-2013", "2004-2013"]):
    # the 2 rows of this window (grid and qualifying), with the position as row name
    window = six[six["before"] == before].set_index("position")
    # keep the two rates and name them after their years
    pair = window[["rate before", "rate 2014-24"]]
    pair.columns = [before, "2014-24"]

    pair.plot(kind="bar", ax=ax, color=["tab:grey", "tab:green"])
    for bars in ax.containers:
        ax.bar_label(bars, fmt="%.0f", fontsize=9)   # write the value on top of each bar

    # difference between the two grid bars, using the same rounded numbers that are written on the bars
    gain = round(pair.loc["grid", "2014-24"]) - round(pair.loc["grid", before])
    ax.set_title(before + " vs 2014-24\n(grid: +" + str(gain) + " points after 2014)")
    ax.set_xlabel("front row by")
    ax.set_ylabel("% of front-row starters on the podium")
    ax.set_ylim(0, 100)
    ax.tick_params(axis="x", rotation=0)
    ax.legend(loc="lower right")

plt.tight_layout()
plt.show()

**2) The circuit ranking: how similar is each "before" window to 2014-24?** Spearman: 1 = same order, 0 = unrelated.

In [ ]:
spearman_table = six.pivot(index="before", columns="position", values="spearman")

ax = spearman_table.plot(kind="bar", color=["tab:blue", "tab:orange"], figsize=(8, 4.5))
ax.axhline(0, color="black", linewidth=1)
for bars in ax.containers:
    ax.bar_label(bars, fmt="%.2f", fontsize=9)   # write the value on top of each bar
ax.set_title("How similar is the circuit ranking to 2014-24?")
ax.set_xlabel('"before 2014" window')
ax.set_ylabel("Spearman (1 = same order, 0 = unrelated)")
ax.set_ylim(-0.5, 1)
ax.tick_params(axis="x", rotation=0)
plt.show()

#### Why did the front row convert more often?

Every front-row starter ends in one of three ways: **podium**, **did not finish** (the car broke down, crashed or was
disqualified), or **finished 4th or lower**. We also count how often both front-row cars came from the same team.
`outcomes` is used here only to explain the result, never as a model feature.

In [ ]:
# add positionText from outcomes: a number ("1", "2", ...) if the driver was classified, a letter (R = retired...) if not
why = q1.merge(outcomes[["raceId", "driverId", "positionText"]], on=["raceId", "driverId"], how="left")
why = why[(why["grid"] == 1) | (why["grid"] == 2)].copy()

# three spans of years
why["span"] = "1950-2003"
why.loc[why["year"] >= 2004, "span"] = "2004-13"
why.loc[why["year"] >= 2014, "span"] = "2014-24"

# put every front-row starter into one of three groups
why["result"] = "finished 4th or lower"
why.loc[~why["positionText"].str.isnumeric(), "result"] = "did not finish"
why.loc[why["podium"] == 1, "result"] = "podium"

# % of each group per span (normalize="index" makes every row add up to 100%)
result_share = pd.crosstab(why["span"], why["result"], normalize="index") * 100
result_share = result_share[["podium", "finished 4th or lower", "did not finish"]]
display(result_share.round(1))

result_share.plot(kind="bar", stacked=True, color=["tab:green", "tab:orange", "tab:red"], figsize=(7, 4))
plt.title("What happened to the front-row starters (grid 1st/2nd)?")
plt.xlabel("years")
plt.ylabel("% of front-row starters")
plt.xticks(rotation=0)
plt.legend(title="result", bbox_to_anchor=(1, 1))
plt.show()

# how often were both front-row cars from the same team?
per_race = why.groupby(["raceId", "span"]).agg(cars=("constructorId", "count"), teams=("constructorId", "nunique"))
per_race = per_race.reset_index()
per_race = per_race[per_race["cars"] == 2]
per_race["same_team"] = per_race["teams"] == 1
print("% of races where both front-row cars were from the same team:")
print((per_race.groupby("span")["same_team"].mean() * 100).round(1))

#### Check: does our choice of 10 starts change the answer?

The minimum of 10 starts is our own decision. We repeat the main comparison (2004-13 vs 2014-24) with 6, 8 and 15
to see whether the answer (how similar the rankings are, and which circuits are on top after 2014) stays the same.

In [ ]:
rows = []
for min_starts in [6, 8, 10, 15]:
    for position_column, table in [("grid", grid_fair), ("qual_or_grid", qual_fair)]:
        rates, spearman = compare_eras(table, "2004-2013", min_starts=min_starts)
        top5 = rates.sort_values("2014-24", ascending=False).head(5).index
        rows.append([position_column, min_starts, len(rates), round(spearman, 2), ", ".join(top5)])

check = pd.DataFrame(rows, columns=["position", "min starts", "circuits", "spearman", "top 5 in 2014-24"])
with pd.option_context("display.max_colwidth", None):
    display(check)

#### Answer to Q1

**1. Best circuits in the hybrid era (2014-24): Monaco and Suzuka.** About 9 in 10 front-row starters finished on the podium
there: Monaco 17 of 19, Suzuka 16 of 18 (89%). They are the **top 2 in all six comparisons**, with grid and with qualifying.
Next come Yas Marina, Barcelona and Spa (82-86%).

**2. Main comparison, 2004-13 vs 2014-24 (the fairest).** The front row converts more often after 2014:

| Front row by | 2004-13 | 2014-24 |
|---|---|---|
| grid (started 1st/2nd) | 69% (254 of 366) | **76%** (345 of 453) |
| qualifying (qualified 1st/2nd) | 69% (251 of 366) | **74%** (336 of 454) |

The ranking is **only partly the same**. Spearman is **0.28 (grid)** and **0.45 (qualifying)**, for 14 circuits.
- **Moved up:** Yas Marina (60% → 86%), Barcelona, Interlagos, Canada.
- **Moved down:** Monza, Silverstone, Bahrain, Hungaroring.
- Monza and Silverstone barely changed their own rate (75% → 73%, 75% → 75% on grid). They dropped because other circuits rose.
- With a minimum of 6, 8 or 15 starts, Spearman stays positive (0.09-0.39 grid, 0.24-0.54 qualifying), and Monaco and Suzuka
  stay the top 2.

**3. The further back "before" goes, the more different the ranking looks.**

| Before | Spearman grid | Spearman qualifying |
|---|---|---|
| 2004-2013 | 0.28 | 0.45 |
| 1994-2013 | -0.04 | -0.00 |
| 1950-2013 | -0.08 | -0.11 |

Compared with the last decade, part of the order is kept. Compared with all of F1 history, the old order says nothing about the new one.

**4. Why the front row converts more often.**
- Over the long run, front-row cars stopped breaking down: **31%** did not finish in 1950-2003, **10%** in 2004-13, **7%** in 2014-24.
- Between 2004-13 and 2014-24, the share that finished but lost the podium on track also fell (20% → 17%).
- Both front-row cars came from the same team more often (32% → 44% of races). One dominant team kept both spots.

**5. Grid vs qualifying.**
- In 1994-2003 the grid was the qualifying order. 98.5% of drivers started where they qualified, and all 164 front-row
  qualifiers started on the front row.
- Grid penalties start in 2004. Drivers per race who started 3+ places behind their qualifying spot: **0.01** (1994-2003),
  **0.76** (2004-13), **1.05** (2014-24)
  (the 3-4 and 5+ columns of the places-lost table added together).
- On the front row, 0 → 9 → 28 qualifiers were pushed back. Example: Verstappen, Belgium 2024, qualified 1st and started 11th.
- This is why the grid and qualifying answers are identical before 2004 and differ a little after.
- Qualifying gives a slightly lower rate after 2014 (74% vs 76%), because a fast car pushed off the front row still counts
  in qualifying.

**Grain**
- Input: one row per driver per race (`model_base`).
- Front-row subset: 2 rows per race (position 1 and 2).
- Result tables: one row per circuit and era.
- The circuits join is many-to-one, and the row count is checked to stay the same.

**Missingness**
- `qual_position` is empty before 1994 and patchy in 1996-2002 (see the coverage chart).
- `qual_or_grid` fills these gaps with the grid, **only before 2004**, when grid = qualifying order (checked: 164 of 164
  front-row qualifiers started on the front row).
- 21 missing values from 2004 on are not filled.
- `grid` has no gaps. `grid = 0` (pit-lane start) is not counted as front row.

**Pre-race cutoff**
- `grid` and `qual_position` are both known before the race starts.
- `podium` is the outcome we measure, not an input.
- `outcomes` (positionText) is used only to explain why the rate rose, never as an input.

**Limitations**
- Some circuits hosted only a few races in an era (fewer than 10 front-row starts), so they are left out of the ranking.
- One `circuitId` covers all layout changes of a track (e.g. Silverstone, Spa).
- Circuits come and go. Magny-Cours and Istanbul topped 2004-13 but are not on the calendar after 2014.
- Shared drives (two drivers, one car) exist only before 1965 and are counted once, with the best finish.

#### Backup: longer "before" windows and four periods

Extra detail, not needed for the answer above.

In [ ]:
# rank change for the two longer windows: 1994-2013 and 1950-2013 (top = grid, bottom = qualifying)
fig, axes = plt.subplots(2, 2, figsize=(15, 14))
for column, start_year in enumerate([1994, 1950]):
    data = add_era(q1, start_year)
    before = str(start_year) + "-2013"
    rates, spearman = compare_eras(front_row_rate(data, "grid"), before)
    slope_chart(axes[0, column], rates, spearman, "Rank change " + before + " -> 2014-24, grid")
    rates, spearman = compare_eras(front_row_rate(data, "qual_or_grid"), before)
    slope_chart(axes[1, column], rates, spearman, "Rank change " + before + " -> 2014-24, qualifying")
plt.tight_layout()
plt.show()

In [ ]:
# four periods: the rate for all circuits together (grid)
q1["period"] = "1950-93"
q1.loc[q1["year"] >= 1994, "period"] = "1994-2003"
q1.loc[q1["year"] >= 2004, "period"] = "2004-13"
q1.loc[q1["year"] >= 2014, "period"] = "2014-24"

grid_front = q1[(q1["grid"] == 1) | (q1["grid"] == 2)]
by_period = grid_front.groupby("period")["podium"].agg(starts="count", podiums="sum")
by_period["rate"] = by_period["podiums"] / by_period["starts"]
display(by_period.round(3))

plt.figure(figsize=(7, 4))
plt.bar(by_period.index, by_period["rate"] * 100, color="tab:blue", label="all circuits together")
plt.ylim(0, 100)
plt.title("Front row (grid 1st/2nd) -> podium, per period")
plt.xlabel("period")
plt.ylabel("% of front-row starters on the podium")
plt.legend(loc="upper left")
plt.show()

In [ ]:
# heatmap: rate per circuit and period, only where the circuit had at least 10 front-row starts
grid_front = q1[(q1["grid"] == 1) | (q1["grid"] == 2)]
period_table = grid_front.groupby(["circuit", "period"])["podium"].agg(starts="count", podiums="sum").reset_index()
period_table["rate"] = period_table["podiums"] / period_table["starts"]
period_table = period_table[period_table["starts"] >= MIN_STARTS]
heat = period_table.pivot(index="circuit", columns="period", values="rate") * 100

# keep circuits that appear in at least 2 periods, best in 2014-24 at the top
heat = heat[heat.notna().sum(axis=1) >= 2]
heat = heat.sort_values("2014-24", ascending=False)

plt.figure(figsize=(8, 9))
sns.heatmap(heat, annot=True, fmt=".0f", cmap="Greens", vmin=0, vmax=100,
            cbar_kws={"label": "% of front-row starters on the podium"})
plt.title("Front row (grid 1st/2nd) -> podium per circuit and period\n(blank = fewer than 10 starts)")
plt.xlabel("period")
plt.ylabel("circuit")
plt.show()